In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

np.random.seed(42)
n = 1000

data = pd.DataFrame({
    "age": np.random.randint(18, 65, n),
    "gender": np.random.choice(["男", "女"], n),
    "browse_duration": np.round(np.random.exponential(30, n), 1),
    "add_to_cart": np.random.poisson(3, n),
    "history_purchases": np.random.randint(0, 50, n),
})
score = (0.03 * data["age"] + 0.5 * (data["gender"] == "女").astype(int)
         + 0.05 * data["browse_duration"] + 0.3 * data["add_to_cart"]
         + 0.1 * data["history_purchases"] - 4)
data["purchased"] = (np.random.random(n) < (1 / (1 + np.exp(-score)))).astype(int)

data["gender_encoded"] = LabelEncoder().fit_transform(data["gender"])
feature_cols = ["age", "gender_encoded", "browse_duration", "add_to_cart", "history_purchases"]
X = StandardScaler().fit_transform(data[feature_cols].values)
y = data["purchased"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [8]:
models = {
    "逻辑回归": LogisticRegression(max_iter=1000, random_state=42),
    "随机森林": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100, use_label_encoder=False, eval_metric="logloss", random_state=42
    ),
}

In [9]:
results = {}

In [10]:
for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    results[name] = acc
    print(f"{'='*40}")
    print(f"📊 {name} | Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["未购买", "已购买"]))

📊 逻辑回归 | Accuracy: 0.8650
              precision    recall  f1-score   support

         未购买       0.75      0.46      0.57        39
         已购买       0.88      0.96      0.92       161

    accuracy                           0.86       200
   macro avg       0.82      0.71      0.75       200
weighted avg       0.86      0.86      0.85       200

📊 随机森林 | Accuracy: 0.8300
              precision    recall  f1-score   support

         未购买       0.59      0.44      0.50        39
         已购买       0.87      0.93      0.90       161

    accuracy                           0.83       200
   macro avg       0.73      0.68      0.70       200
weighted avg       0.82      0.83      0.82       200

📊 XGBoost | Accuracy: 0.8350
              precision    recall  f1-score   support

         未购买       0.61      0.44      0.51        39
         已购买       0.87      0.93      0.90       161

    accuracy                           0.83       200
   macro avg       0.74      0.68      0.70    

E:\anaconda3\envs\MachineLearning\lib\site-packages\xgboost\training.py:200: UserWarning: [22:24:25] WARNING: C:\Users\task_177929407859648\croot\xgboost-split_1779294268734\work\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [11]:
print("\n模型准确率排名:")
for rank, (name, acc) in enumerate(sorted(results.items(), key=lambda x: -x[1]), 1):
    print(f"  {rank}. {name}: {acc:.4f}")

best_name = max(results, key=results.get)
print(f"\n最佳基线模型: {best_name} ({results[best_name]:.4f})")


模型准确率排名:
  1. 逻辑回归: 0.8650
  2. XGBoost: 0.8350
  3. 随机森林: 0.8300

最佳基线模型: 逻辑回归 (0.8650)
